In [ ]:
# Préparation commune des TP Python
# Le notebook utilise uniquement les ressources fournies dans le dossier codes/.
from pathlib import Path
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib'}
missing = [package for module, package in REQUIRED_PACKAGES.items()
           if importlib.util.find_spec(module) is None]
if missing:
    print("Installation des paquets manquants :", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

WORKDIR = Path.cwd().resolve()
CODES_DIR = WORKDIR.parent if WORKDIR.name == "correction" else WORKDIR
TOOLBOX_DIR = CODES_DIR / "toolbox"
DATA_DIR = CODES_DIR / "data"
if not CODES_DIR.is_dir():
    raise FileNotFoundError("Exécutez le notebook depuis le dossier codes/." )
if TOOLBOX_DIR.is_dir():
    sys.path.insert(0, str(TOOLBOX_DIR))

DATA_FILE = DATA_DIR / "temperature_historique.csv"
if not DATA_FILE.is_file():
    raise FileNotFoundError(
        f"Donnée du cours introuvable : {DATA_FILE}. "
        "Téléchargez le dossier codes complet, avec data/."
    )

print("Environnement prêt :", CODES_DIR)


# TP1 — Données climatiques historiques et tendances

**Date de la séance :** mercredi 2 septembre 2026, 08:30–10:30

**Objectifs**
- Manipuler une série temporelle de température globale
- Estimer une tendance de réchauffement climatique
- Distinguer variabilité naturelle et tendance de long terme
- Projeter la date de dépassement de seuils de réchauffement (+1,5°C, +2°C)

**Prérequis** : numpy, pandas, matplotlib (bases de manipulation de tableaux et de graphiques).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Partie 1 — Chargement des données

In [ ]:
df = pd.read_csv(DATA_DIR / "temperature_historique.csv")
print(df.shape)
df.head()

**Question 1.** Tracez l'anomalie de température de 1880 à 2023.

In [ ]:
# Expected output: courbe bruitée croissante de -0.2°C (1880) à environ +1.1°C (2023)
plt.figure(figsize=(9, 5))
plt.plot(df['year'], df['T_anomaly'], color='tab:blue')
plt.xlabel('Année')
plt.ylabel('Anomalie de température (°C)')
plt.title('Anomalie de température globale 1880-2023')
plt.grid(True)
plt.show()

## Partie 2 — Tendance linéaire

**Question 2.** Calculez la tendance linéaire avec `np.polyfit`. Ajoutez la droite de tendance au graphique.

In [ ]:
# Expected output: pente positive d'environ 0.007-0.008 °C/an (~0.7-0.8°C/siècle)
coeffs = np.polyfit(df['year'], df['T_anomaly'], 1)
trend = np.polyval(coeffs, df['year'])
print(f"Pente estimée : {coeffs[0]:.5f} °C/an  (soit {coeffs[0]*100:.2f} °C/siècle)")

plt.figure(figsize=(9, 5))
plt.plot(df['year'], df['T_anomaly'], color='tab:blue', label='Anomalie observée')
plt.plot(df['year'], trend, color='tab:red', linewidth=2, label='Tendance linéaire')
plt.xlabel('Année')
plt.ylabel('Anomalie de température (°C)')
plt.title('Anomalie de température et tendance linéaire')
plt.legend()
plt.grid(True)
plt.show()

## Partie 3 — Variabilité naturelle et tendance longue

**Question 3.** Calculez une moyenne mobile sur 10 ans. Tracez l'anomalie, la tendance et la moyenne mobile.

In [ ]:
# Expected output: la moyenne mobile lisse le bruit annuel et suit la tendance de long terme
window = 10
df['rolling'] = df['T_anomaly'].rolling(window, center=True).mean()

plt.figure(figsize=(9, 5))
plt.plot(df['year'], df['T_anomaly'], color='tab:blue', alpha=0.4, label='Anomalie observée')
plt.plot(df['year'], trend, color='tab:red', linewidth=2, label='Tendance linéaire')
plt.plot(df['year'], df['rolling'], color='tab:green', linewidth=2, label=f'Moyenne mobile {window} ans')
plt.xlabel('Année')
plt.ylabel('Anomalie de température (°C)')
plt.title('Anomalie, tendance et variabilité naturelle')
plt.legend()
plt.grid(True)
plt.show()

## Partie 4 — Moyennes décennales

**Question 4.** Calculez la température moyenne par décennie (1880-1889, 1890-1899, ..., 2010-2019). Tracez un graphique en barres.

In [ ]:
# Expected output: barres croissantes par décennie, de ~-0.15°C (1880s) à ~+0.9°C (2010s)
df['decade'] = (df['year'] // 10) * 10
dec_avg = df[df['decade'] < 2020].groupby('decade')['T_anomaly'].mean()

plt.figure(figsize=(9, 5))
dec_avg.plot(kind='bar', color='tab:orange')
plt.xlabel('Décennie')
plt.ylabel('Anomalie moyenne (°C)')
plt.title('Anomalie de température moyenne par décennie')
plt.grid(True, axis='y')
plt.tight_layout()
plt.show()
dec_avg

## Partie 5 — Projection

**Question 5.** En prolongeant la tendance linéaire, en quelle année atteint-on +1,5°C ? +2°C ?

In [ ]:
# Expected output: années de dépassement obtenues par extrapolation linéaire (résultat purement illustratif,
# une vraie projection climatique nécessite un modèle physique, cf. TP2)
year_15 = (1.5 - coeffs[1]) / coeffs[0]
year_20 = (2.0 - coeffs[1]) / coeffs[0]
print(f"Seuil de +1.5°C atteint (extrapolation linéaire) vers l'année {year_15:.0f}")
print(f"Seuil de +2.0°C atteint (extrapolation linéaire) vers l'année {year_20:.0f}")

## Interprétation

**Répondez en quelques phrases :**
1. Quelle est la vitesse de réchauffement observée depuis 1880 ?
2. Est-ce que la tendance s'accélère ou reste-t-elle constante ?
3. Quelle est la différence entre variabilité naturelle et tendance climatique ?
4. Que représente l'anomalie de température par rapport à la météo quotidienne ?

### Éléments de réponse

1. La tendance linéaire estimée sur 1880-2023 correspond à environ 0,7-0,8°C de réchauffement par siècle.
2. Les données synthétiques utilisées ici suivent une tendance linéaire par construction ; sur les données réelles, le réchauffement observé s'est nettement accéléré depuis les années 1970-1980 (la tendance récente est supérieure à la tendance sur tout le XXe siècle).
3. La variabilité naturelle correspond aux fluctuations d'une année sur l'autre (cycles océaniques comme El Niño/La Niña, éruptions volcaniques, variabilité interne du système climatique) ; la tendance climatique est le signal de fond, robuste sur plusieurs décennies, attribuable au forçage radiatif anthropique.
4. L'anomalie de température est un écart par rapport à une période de référence (moyenne climatologique), et non une température absolue journalière : elle filtre les effets saisonniers et géographiques pour isoler le signal de changement climatique global.